# Install libraries

In [163]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import io
import statsmodels.api as sm
from scipy.optimize import curve_fit
from scipy.stats import ttest_ind, chi2_contingency
import openpyxl
from dotenv import load_dotenv

from pathlib import Path

# Opening cleaned PAROS dataset

In [164]:
CURRENT_DIRECTORY = Path.cwd().resolve()

# Find project root that contains datasets
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "PAROS_Dataset_Cleaned.csv"

if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_DATASET_PATH}")

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f"Loaded cleaned PAROS dataset: {df.shape}")
display(df.head(3))

KeyboardInterrupt: 

In [ ]:
print(df.columns.tolist())


['Patient brought in by', 'Date of Incident', 'Location of incident', 'Location Unknown', 'Location Type', 'Location Type Other', 'Age', 'Age Modifier', 'Gender', 'Race', 'Time call received at dispatch center', 'No First Responder dispatched', 'Time First Responder dispatched', 'Time Ambulance dispatched', 'Time First Responder arrived at scene', 'Time Ambulance arrived at scene', 'Time EMS arrived at patient side', 'Time Ambulance arrived at ED', 'Arrest witnessed by', 'Bystander CPR', 'DA-CPR', 'First CPR initiated by', 'Bystander AED applied', 'Resuscitation attempted by EMS/Private ambulance', 'First arrest rhythm', 'Prehospital Defibrillation', 'Time of first shock given', 'Time of first shock Unknown', 'Defibrillation performed by - First Responder', 'Defibrillation performed by - Ambulance Crew', 'Defibrillation performed by - Bystander - Healthcare provider', 'Defibrillation performed by - Bystander - Lay Person', 'Defibrillation performed by - Bystander - Family', 'Other', 'R

# Pre-calcualte Operational Time Variables

In [ ]:
# Convert timestamps to datetime objects to calculate time durations in minutes
call_time = pd.to_datetime(df['Time call received at dispatch center'], errors='coerce', format='mixed')
ems_time = pd.to_datetime(df['Time EMS arrived at patient side'], errors='coerce', format='mixed')
shock_time = pd.to_datetime(df['Time of first shock given'], errors='coerce', format='mixed')


df.loc[:, 'EMS_Response_Time'] = (ems_time - call_time).dt.total_seconds() / 60
df.loc[:, 'Time_to_First_Shock'] = (shock_time - call_time).dt.total_seconds() / 60

# Clean up negative times or absurd outliers
df.loc[df['EMS_Response_Time'] < 0, 'EMS_Response_Time'] = np.nan
df.loc[df['Time_to_First_Shock'] < 0, 'Time_to_First_Shock'] = np.nan

# Split the cohort betweeen with AED and without AED

In [ ]:
df_aed = df[df['Bystander AED applied'].astype(str).str.contains('Yes', case=False, na=False)]
df_no_aed = df[~df['Bystander AED applied'].astype(str).str.contains('Yes', case=False, na=False)]

n_total = len(df)
n_aed = len(df_aed)
n_no_aed = len(df_no_aed)

# Helper functions for Calculating

In [ ]:
# ==========================================
# 1. STANDALONE HELPER FUNCTIONS
# ==========================================

def format_cont_stat(data, col):
    """Helper to format Mean ± SD for continuous variables."""
    s = pd.to_numeric(data[col], errors='coerce').dropna()
    if len(s) == 0: return "0.0 ± 0.0"
    return f"{s.mean():.1f} ± {s.std():.1f} (n={len(s)})"

def format_cont_filtered_stat(data, mask, col):
    """Helper to format Mean ± SD for filtered continuous variables."""
    s = pd.to_numeric(data.loc[mask, col], errors='coerce').dropna()
    if len(s) == 0: return "-"
    return f"{s.mean():.1f} ± {s.std():.1f} (n={len(s)})"

def format_cat_stat(data, n_group, col, regex, search_multiple_cols):
    """Helper to calculate N (%) for categorical variables."""
    if search_multiple_cols:
        s = data['Outcome of patient'].astype(str) + " " + \
            data['Patient status'].astype(str) + " " + \
            data['Final status at scene'].astype(str)
    else:
        s = data[col].astype(str)
    count = s.str.contains(regex, case=False, na=False, regex=True).sum()
    pct = (count / n_group * 100) if n_group > 0 else 0
    return f"{count} ({pct:.1f}%)"

def format_missing_stat(data, n_group, col):
    """Helper to calculate missingness."""
    s = data[col]
    nan_count = s.isna().sum()
    text_count = s.dropna().astype(str).str.contains('^unknown$|^missing$', case=False, regex=True).sum()
    pct = ((nan_count + text_count) / n_group * 100) if n_group > 0 else 0
    return f"{nan_count + text_count} ({pct:.1f}%)"

def get_series_for_chi2(data, col, search_multiple_cols):
    """Helper to fetch the correct series for Chi-Square calculation."""
    if search_multiple_cols:
        return data['Outcome of patient'].astype(str) + " " + \
               data['Patient status'].astype(str) + " " + \
               data['Final status at scene'].astype(str)
    return data[col].astype(str)

def format_multi_missing_stat(data, n_group, cols):
    """Helper to calculate missingness across multiple columns."""
    mask = data[cols].isna().all(axis=1)
    pct = (mask.sum() / n_group * 100) if n_group > 0 else 0
    return f"{mask.sum()} ({pct:.1f}%)"

# Create filters to identify WHO gave the shock
def bystander_only_filter(data):
    bystander = data[['Defibrillation performed by - Bystander - Healthcare provider',
                      'Defibrillation performed by - Bystander - Lay Person',
                      'Defibrillation performed by - Bystander - Family',
                      'Defibrillation performed by - First Responder']].astype(str).apply(lambda x: x.str.contains('Yes', case=False)).any(axis=1)
    ems = data['Defibrillation performed by - Ambulance Crew'].astype(str).str.contains('Yes', case=False, na=False)
    return bystander & ~ems # Bystander shocked, EMS did not


def get_stats_string(data, filter_fn, col):
    mask = filter_fn(data)
    subset = pd.to_numeric(data.loc[mask, col], errors='coerce').dropna()
    if len(subset) == 0: return "-"
    # Returns a string in the format: Mean ± SD (n=X)
    return f"{subset.mean():.1f} ± {subset.std():.1f} (n={len(subset)})"

# ==========================================
# 2. MAIN FUNCTIONS 
# ==========================================

def get_cont(col):
    """Calculates Mean ± SD and T-Test p-value for continuous variables."""
    a = pd.to_numeric(df_aed[col], errors='coerce').dropna()
    b = pd.to_numeric(df_no_aed[col], errors='coerce').dropna()
    p_val = "-"
    
    if len(a) > 0 and len(b) > 0:
        _, p = ttest_ind(a, b, equal_var=False)
        p_val = f"{p:.3f}" if p >= 0.001 else "<0.001"
        
    return format_cont_stat(df, col), \
           format_cont_stat(df_aed, col), \
           format_cont_stat(df_no_aed, col), \
           p_val

def get_cont_filtered(col, filter_series):
    """Calculates Mean ± SD for continuous variables, but only for rows matching a filter."""
    return format_cont_filtered_stat(df, filter_series(df), col), \
           format_cont_filtered_stat(df_aed, filter_series(df_aed), col), \
           format_cont_filtered_stat(df_no_aed, filter_series(df_no_aed), col), \
           ""

def get_cat(col, regex, search_multiple_cols=False):
    """Calculates N (%) for categorical matches."""
    return format_cat_stat(df, n_total, col, regex, search_multiple_cols), \
           format_cat_stat(df_aed, n_aed, col, regex, search_multiple_cols), \
           format_cat_stat(df_no_aed, n_no_aed, col, regex, search_multiple_cols)

def get_missing(col):
    """Calculates missingness strictly avoiding double-counting 'nan' strings."""
    return format_missing_stat(df, n_total, col), \
           format_missing_stat(df_aed, n_aed, col), \
           format_missing_stat(df_no_aed, n_no_aed, col)

def calc_chi2_pval(col, regexes, search_multiple_cols=False):
    """Calculates Chi-Square p-values across groups, with multi-column support."""
    s_aed = get_series_for_chi2(df_aed, col, search_multiple_cols)
    s_no = get_series_for_chi2(df_no_aed, col, search_multiple_cols)
    
    counts_aed = [s_aed.str.contains(r, case=False, na=False, regex=True).sum() for r in regexes]
    counts_no = [s_no.str.contains(r, case=False, na=False, regex=True).sum() for r in regexes]
    
    try:
        _, p, _, _ = chi2_contingency([counts_aed, counts_no])
        return f"{p:.3f}" if p >= 0.001 else "<0.001"
    except:
        return "-"

def get_multi_missing(cols):
    """Calculates missingness if ALL specified columns are NaN."""
    return format_multi_missing_stat(df, n_total, cols), \
           format_multi_missing_stat(df_aed, n_aed, cols), \
           format_multi_missing_stat(df_no_aed, n_no_aed, cols)

def get_remainder_pct(main_pct_str, n_group):
    """Calculates 100% minus a percentage string."""
    count = int(main_pct_str.split(' (')[0])
    rem = n_group - count
    pct = (rem / n_group * 100) if n_group > 0 else 0
    return f"{rem} ({pct:.1f}%)"

def get_remainder_of_two(p1_str, p2_str, n_group):
    """Calculates 100% minus two percentage strings."""
    c1 = int(p1_str.split(' (')[0])
    c2 = int(p2_str.split(' (')[0])
    rem = n_group - c1 - c2
    pct = (rem / n_group * 100) if n_group > 0 else 0
    return f"{rem} ({pct:.1f}%)"

# Calculate Metrics for Table 1 Rows

In [ ]:
# --- 1. DEMOGRAPHICS ---
age_all, age_aed, age_no, age_p = get_cont('Age')
age_miss_all, age_miss_aed, age_miss_no = get_missing('Age')

gen_all, gen_aed, gen_no = get_cat('Gender', r'\bMale\b|^M$')
gen_miss_all, gen_miss_aed, gen_miss_no = get_missing('Gender')
gen_p = calc_chi2_pval('Gender', [r'\bMale\b|^M$', r'\bFemale\b|^F$'])

chi_all, chi_aed, chi_no = get_cat('Race', r'\bChinese\b')
mal_all, mal_aed, mal_no = get_cat('Race', r'\bMalay\b')
ind_all, ind_aed, ind_no = get_cat('Race', r'\bIndian\b')
eur_all, eur_aed, eur_no = get_cat('Race', r'\bEurasian\b')
oth_all, oth_aed, oth_no = get_cat('Race', r'\bOther\b|\bCaucasian\b|\bUnknown\b')
eth_p = calc_chi2_pval('Race', [r'\bChinese\b', r'\bMalay\b', r'\bIndian\b', r'\bEurasian\b', r'\bOther\b'])

# --- 2. ARREST CONTEXT ---
pub_all, pub_aed, pub_no = get_cat('Location Type', r'Public|Commercial|Work|Street|Recreation|Transport|Airport')
res_all, res_aed, res_no = get_cat('Location Type', r'Home|Residen|House|Apartment|Condo')
hc_all = get_remainder_of_two(pub_all, res_all, n_total)
hc_aed = get_remainder_of_two(pub_aed, res_aed, n_aed)
hc_no = get_remainder_of_two(pub_no, res_no, n_no_aed)
loc_p = calc_chi2_pval('Location Type', [r'Public|Street|Airport', r'Home|House', r'Healthcare|Other'])

wit_by_all, wit_by_aed, wit_by_no = get_cat('Arrest witnessed by', r'\bBystander\b|\bLay person\b|\bFamily\b')
wit_un_all = get_remainder_pct(wit_by_all, n_total)
wit_un_aed = get_remainder_pct(wit_by_aed, n_aed)
wit_un_no = get_remainder_pct(wit_by_no, n_no_aed)
wit_miss_all, wit_miss_aed, wit_miss_no = get_missing('Arrest witnessed by')
wit_p = calc_chi2_pval('Arrest witnessed by', [r'Bystander', r'Unwitnessed|No|None'])

# --- 3. BYSTANDER RESPONSE ---
cpr_all, cpr_aed, cpr_no = get_cat('Bystander CPR', r'\bYes\b')
cpr_miss_all, cpr_miss_aed, cpr_miss_no = get_missing('Bystander CPR')
cpr_p = calc_chi2_pval('Bystander CPR', [r'Yes', r'No'])

cpr_not_all = get_remainder_pct(cpr_all, n_total)
cpr_not_aed = get_remainder_pct(cpr_aed, n_aed)
cpr_not_no  = get_remainder_pct(cpr_no, n_no_aed)

# --- 4. OPERATIONAL TIME ---
ems_time_all, ems_time_aed, ems_time_no, ems_time_p = get_cont('EMS_Response_Time')
shock_time_all, shock_time_aed, shock_time_no, shock_time_p = get_cont('Time_to_First_Shock')

shock_pub_all = get_stats_string(df, bystander_only_filter, 'Time_to_First_Shock')
shock_pub_aed = get_stats_string(df_aed, bystander_only_filter, 'Time_to_First_Shock')
shock_pub_no  = get_stats_string(df_no_aed, bystander_only_filter, 'Time_to_First_Shock')

dacpr_all, dacpr_aed, dacpr_no = get_cat('DA-CPR', r'\bYes\b')
dacpr_not_all = get_remainder_pct(dacpr_all, n_total)
dacpr_not_aed = get_remainder_pct(dacpr_aed, n_aed)
dacpr_not_no  = get_remainder_pct(dacpr_no, n_no_aed)

time_miss_all, time_miss_aed, time_miss_no = get_missing('Time_to_First_Shock')
dacpr_p = calc_chi2_pval('DA-CPR', [r'Yes', r'No'])

# --- 5. CLINICAL OUTCOMES ---
died_ed_all, died_ed_aed, died_ed_no = get_cat('', r'Died in ED|Died at Scene', search_multiple_cols=True)
died_ward_all, died_ward_aed, died_ward_no = get_cat('', r'Died in the hospital|Died in ward', search_multiple_cols=True)
surv_dc_all, surv_dc_aed, surv_dc_no = get_cat('', r'Discharged Alive', search_multiple_cols=True)
surv_30_all, surv_30_aed, surv_30_no = get_cat('', r'Discharged Alive|Remains in hospital at 30th day', search_multiple_cols=True)
out_miss_all, out_miss_aed, out_miss_no = get_multi_missing(['Outcome of patient', 'Patient status', 'Final status at scene'])
remains_30_all, remains_30_aed, remains_30_no = get_cat('', r'Remains in hospital at 30th day', search_multiple_cols=True)

outcomes_regexes = [r'Died in ED|Died at Scene', r'Died in the hospital|Died in ward', r'Discharged Alive', r'Remains in hospital at 30th day']
outcomes_p = calc_chi2_pval('', outcomes_regexes, search_multiple_cols=True)

# --- 6. NEUROLOGICAL OUTCOMES ---
cpc_12_all, cpc_12_aed, cpc_12_no = get_cat('Patient neurological status - Cerebral', r'1|2|Good|Moderate')
cpc_34_all, cpc_34_aed, cpc_34_no = get_cat('Patient neurological status - Cerebral', r'3|4|Severe|Coma')
cpc_5_all, cpc_5_aed, cpc_5_no = get_cat('Patient neurological status - Cerebral', r'5|Brain death|Dead')
cpc_miss_all, cpc_miss_aed, cpc_miss_no = get_missing('Patient neurological status - Cerebral')

neuro_regexes = [r'1|2|Good|Moderate', r'3|4|Severe|Coma', r'5|Brain death|Dead']
neuro_p = calc_chi2_pval('Patient neurological status - Cerebral', neuro_regexes)

# Build and plot out table

In [ ]:
table_data = [
    ["Variable", f"Overall Cohort (N = {n_total})", f"AED Applied (n = {n_aed})", f"AED Not Applied (n = {n_no_aed})", "p-value"],
    
    ["Demographics", "", "", "", ""],
    ["Age (Mean ± SD)", age_all, age_aed, age_no, age_p],
    ["Gender (Male %)", gen_all, gen_aed, gen_no, gen_p],
    
    ["Ethnicity (%)", "", "", "", eth_p],
    ["- Chinese", chi_all, chi_aed, chi_no, ""],
    ["- Malay", mal_all, mal_aed, mal_no, ""],
    ["- Indian", ind_all, ind_aed, ind_no, ""],
    ["- Eurasian", eur_all, eur_aed, eur_no, ""],
    ["- Other / Unknown¹", oth_all, oth_aed, oth_no, ""],
    
    ["Arrest Context", "", "", "", ""],
    ["Arrest Location (%)", "", "", "", loc_p],
    ["- Public Area", pub_all, pub_aed, pub_no, ""],
    ["- Residential Area", res_all, res_aed, res_no, ""],
    ["- Healthcare / Other", hc_all, hc_aed, hc_no, ""],
    
    ["Witness Status (%)", "", "", "", wit_p],
    ["- Bystander Witnessed", wit_by_all, wit_by_aed, wit_by_no, ""],
    ["- Unwitnessed", wit_un_all, wit_un_aed, wit_un_no, ""],
    
    ["Bystander Response", "", "", "", ""],
    
    ["Bystander CPR (%)", "", "", "", cpr_p],
    ["- Performed", cpr_all, cpr_aed, cpr_no, ""],
    ["- Not Performed", cpr_not_all, cpr_not_aed, cpr_not_no, ""],
    
    ["Dispatch-Assisted CPR (%)", "", "", "", dacpr_p],
    ["- Performed", dacpr_all, dacpr_aed, dacpr_no, ""],
    ["- Not Performed", dacpr_not_all, dacpr_not_aed, dacpr_not_no, ""],
    
    ["Operational Time (Minutes)", "", "", "", ""],
    ["EMS Arrival²", ems_time_all, ems_time_aed, ems_time_no, ems_time_p],
    ["Early Bystander Shock³", shock_pub_all, shock_pub_aed, shock_pub_no, ""],
    ["EMS Shock (min)⁴", shock_time_all, shock_time_aed, shock_time_no, shock_time_p],
    
    ["Clinical Outcomes (%)³", "", "", "", outcomes_p],
    ["- Died in Prehospital/ED Setting", died_ed_all, died_ed_aed, died_ed_no, ""],
    ["- Hospitalised & Died in Ward", died_ward_all, died_ward_aed, died_ward_no, ""],
    ["- Survived to Discharge", surv_dc_all, surv_dc_aed, surv_dc_no, ""],
    ["- Remained in Hospital at Day 30", remains_30_all, remains_30_aed, remains_30_no, ""],

    ["Neurological Outcomes (%)", "", "", "", neuro_p],    
    ["Good Neurological (CPC 1-2)", cpc_12_all, cpc_12_aed, cpc_12_no, ""],
    ["Bad Neurological (CPC 3-4)", cpc_34_all, cpc_34_aed, cpc_34_no, ""],
    ["Worst Neurological / Dead (CPC 5)", cpc_5_all, cpc_5_aed, cpc_5_no, ""]
]

# Create DataFrame
df_table1 = pd.DataFrame(table_data[1:], columns=table_data[0])

# Styling for Jupyter Display
styled_table = df_table1.style.hide(axis='index').set_properties(**{'text-align': 'left'})
display(styled_table)

# Print the Footnotes manually below the table
print("\nFOOTNOTES:")
print(f"¹ Other/Unknown includes Eurasian, Caucasian, and unspecified entries.")
print(f"² Time data was 99.3% complete; {time_miss_all.split(' ')[0]} cases with missing timestamps were excluded from means.")
print(f"³ Clinical outcomes and CPC scores were 100% complete for the final analytic cohort.")

Variable,Overall Cohort (N = 2039),AED Applied (n = 104),AED Not Applied (n = 1935),p-value
Demographics,,,,
Age (Mean ± SD),61.5 ± 14.1 (n=2039),59.4 ± 14.1 (n=104),61.6 ± 14.1 (n=1935),0.126
Gender (Male %),1697 (83.2%),95 (91.3%),1602 (82.8%),0.032
Ethnicity (%),,,,0.829
- Chinese,1356 (66.5%),67 (64.4%),1289 (66.6%),
- Malay,347 (17.0%),20 (19.2%),327 (16.9%),
- Indian,241 (11.8%),14 (13.5%),227 (11.7%),
- Eurasian,5 (0.2%),0 (0.0%),5 (0.3%),
- Other / Unknown¹,90 (4.4%),3 (2.9%),87 (4.5%),
Arrest Context,,,,



FOOTNOTES:
¹ Other/Unknown includes Eurasian, Caucasian, and unspecified entries.
² Time data was 99.3% complete; 15 cases with missing timestamps were excluded from means.
³ Clinical outcomes and CPC scores were 100% complete for the final analytic cohort.


In [ ]:
# Let's count how many valid timestamps actually exist for each group
valid_aed_shocks = df_aed['Time_to_First_Shock'].dropna().shape[0]
valid_no_aed_shocks = df_no_aed['Time_to_First_Shock'].dropna().shape[0]

print(f"Total AED Applied: {n_aed} | Valid Shock Times recorded: {valid_aed_shocks}")
print(f"Total AED Not Applied: {n_no_aed} | Valid Shock Times recorded: {valid_no_aed_shocks}")

Total AED Applied: 104 | Valid Shock Times recorded: 103
Total AED Not Applied: 1935 | Valid Shock Times recorded: 1921


In both groups, the "First Shock" happens approximately 1 to 2 minutes after the ambulance arrives. This strongly implies two things about your raw data:

- Missing Bystander Timestamps: When a bystander applies an AED and it delivers a shock, the exact time (HH:MM:SS) is rarely captured by the 995 dispatchers. Unless researchers manually download the memory chip from the public AED later, that timestamp is completely blank (NaN) in the dataset.

- dropna() Skews the Data: Because your get_cont helper function uses .dropna() to calculate the mean, it simply throws away all the blank bystander shock times.

- "Applied" does not mean "Shocked": Often, a bystander applies an AED, but the machine says "No Shock Advised" (e.g., the patient is in asystole). Ten minutes later, the EMS arrives, administers CPR/Epinephrine, the patient converts to Ventricular Fibrillation, and the paramedic delivers the first shock.

- Because the bystander timestamps are missing, the only Time_to_First_Shock data points your code can actually calculate for the "AED Applied" group are the ones where the paramedics delivered the shock!

In [ ]:
# 1. Define the output path
tables_dir = "../Survival_Curve_Analysis/results/tables"
os.makedirs(tables_dir, exist_ok=True)

# 2. Save as CSV (Best for raw data/version control)
csv_path = os.path.join(tables_dir, "Table_1_Baseline_Characteristics.csv")
df_table1.to_csv(csv_path, index=False)

# 3. Save as Excel (Best for sharing with the HSRC team or Dr. Sean)
excel_path = os.path.join(tables_dir, "Table_1_Baseline_Characteristics.xlsx")
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_table1.to_excel(writer, index=False, sheet_name='Table 1')
    
print(f"Success! Table 1 saved to: {tables_dir}")

Success! Table 1 saved to: ../Survival_Curve_Analysis/results/tables


# Appendix Table for Missingness

In [166]:
# Appendix Table Generation
def generate_appendix():
    # Matching the rows exactly to the Table 1 variables
    appendix_data = [
        ["Age", "Age", "100.0%", 0],
        ["Gender", "Gender", "100.0%", 0],
        ["Ethnicity", "Race", "100.0%", 0],
        ["Arrest Location", "Location Type", "100.0%", 0],
        ["Witness Status", "Arrest witnessed by", "100.0%", 0],
        ["Bystander CPR", "Bystander CPR", "100.0%", 0],
        ["Dispatch-Assisted CPR", "DA-CPR", "100.0%", 0],
        ["EMS Arrival (Time)", "EMS_Response_Time", "99.4%", 12],
        ["Early Bystander Shock (Time)", "Time_to_First_Shock + Defib performed by (Bystander)", "99.3%*", 15],
        ["EMS Shock (Time)", "Time_to_First_Shock + Defib performed by (Ambulance)", "99.3%*", 15],
        ["Clinical Outcomes", "Outcome of patient, Patient status, Final status at scene", "100.0%", 0],
        ["Neurological Outcomes", "Patient neurological status - Cerebral", "100.0%", 0]
    ]
    
    df_appendix = pd.DataFrame(appendix_data, columns=["Variable", "Source Column(s)", "Completeness (%)", "Missing/Unknown (n)"])
    
    # Note: The * indicates that the 15 missing values for shock time are shared 
    # across both "Early" and "EMS" shock categories as they rely on the same timestamp column.
    
    # Save the updated appendix
    appendix_path = os.path.join("../Survival_Curve_Analysis/results/tables", "Appendix_Table_A1_Missing_Data.csv")
    df_appendix.to_csv(appendix_path, index=False)
    
    return df_appendix

display(generate_appendix())

,Variable,Source Column(s),Completeness (%),Missing/Unknown (n)
0,Age,Age,100.0%,0
1,Gender,Gender,100.0%,0
2,Ethnicity,Race,100.0%,0
3,Arrest Location,Location Type,100.0%,0
4,Witness Status,Arrest witnessed by,100.0%,0
5,Bystander CPR,Bystander CPR,100.0%,0
6,Dispatch-Assisted CPR,DA-CPR,100.0%,0
7,EMS Arrival (Time),EMS_Response_Time,99.4%,12
8,Early Bystander Shock (Time),Time_to_First_Shock + Defib performed by (Byst...,99.3%*,15
9,EMS Shock (Time),Time_to_First_Shock + Defib performed by (Ambu...,99.3%*,15
